# 1a — Install Stage 1 native system dependencies

Run this after `01_setup_and_preflight.ipynb` and before the LIBERO-Plus asset installer. Python's `Wand` package is only a wrapper; LIBERO-Plus also requires the native ImageMagick/MagickWand libraries. MuJoCo headless rendering requires EGL/OSMesa libraries.

This notebook requires passwordless `sudo`, which is normally enabled on the temporary A100 machine. It stops without prompting if that permission is unavailable.

In [ ]:
import subprocess
from pathlib import Path

ood_python = Path.home() / "venv-stage1-ood/bin/python"
if not ood_python.exists():
    raise SystemExit(f"STOP: {ood_python} is missing; complete notebook 1 first")

sudo = subprocess.run(
    ["sudo", "-n", "true"],
    text=True,
    capture_output=True,
)
if sudo.returncode != 0:
    raise SystemExit(
        "STOP: passwordless sudo is unavailable. Ask the machine administrator "
        "to install libmagickwand-dev and the EGL/OSMesa development libraries.\n"
        + sudo.stderr
    )
print("PASS: passwordless sudo available")


In [ ]:
packages = [
    "imagemagick",
    "libmagickwand-dev",
    "libgl1-mesa-dev",
    "libegl1-mesa-dev",
    "libglew-dev",
    "libosmesa6-dev",
    "libglib2.0-0",
    "libexpat1",
    "libfontconfig1-dev",
    "patchelf",
]
subprocess.run(["sudo", "-n", "apt-get", "update"], check=True)
subprocess.run(
    ["sudo", "-n", "apt-get", "install", "-y", *packages],
    check=True,
)
print("Installed:", ", ".join(packages))


In [ ]:
checks = {
    "MagickWand": "from wand.api import library; print('MagickWand import OK')",
    "LIBERO-Plus import": "from pathlib import Path; import os; os.environ['MPLBACKEND']='Agg'; import libero; print('libero namespace OK')",
}
env = __import__("os").environ.copy()
env["PYTHONPATH"] = str(Path.home() / "LIBERO-plus")
env["MPLBACKEND"] = "Agg"
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"
for name, code in checks.items():
    print(f"Checking {name}...")
    subprocess.run([str(ood_python), "-c", code], env=env, check=True)

subprocess.run(["ldconfig", "-p"], stdout=subprocess.DEVNULL, check=True)
print("PASS: Stage 1 native system dependencies are ready")
print("You may now continue with 01b_install_libero_plus_assets.ipynb")
